# Despliegue en *Linux* por *SSH*.

No todo destino es un servicio gestionado. En muchos equipos la aplicación todavía corre en un servidor *Linux* al que se llega por *SSH* (*Secure Shell*). Este cuaderno practica un **primer despliegue remoto controlado**: autenticación por clave, verificación del anfitrión, actualización del servicio y una comprobación posterior.

El entorno de trabajo es un *GitHub Codespace*. Un *runner* hospedado por *GitHub* **no** puede alcanzar un *Codespace*, así que el valor por defecto de esta práctica no usa esos *runners*.

## Objetivos.

* Preparar, dentro del *Codespace*, un destino *Linux* (un contenedor con servidor *SSH*) y un *runner* autoalojado, sin incrustar credenciales en el repositorio.
* Leer el *workflow* ```.github/workflows/despliega-linux.yml``` y ejecutarlo contra ese destino.
* Distinguir el valor por defecto (todo ocurre en el *Codespace*) de la alternativa con un destino *SSH* proporcionado por quien imparte el curso.


## Valor por defecto de la práctica, y cómo cambiarlo.

**Valor por defecto.** En el mismo *Codespace* se levantan dos piezas:

1. un contenedor *Linux* con un servidor *SSH*, que es el destino del despliegue;
2. el *runner* autoalojado del cuaderno 10, que ejecuta el *workflow*.

Como *runner* y destino comparten el *Codespace*, el *job* usa ```runs-on: self-hosted``` y se conecta al contenedor por *SSH* (en la práctica, al anfitrión ```127.0.0.1``` y al puerto publicado por ese contenedor). No hace falta exponer *SSH* a internet.

El archivo vigente ```.github/workflows/despliega-linux.yml``` ya está escrito así.

**Alternativa.** Si quien imparte el curso proporciona un destino *SSH* alcanzable desde los *runners* hospedados de *GitHub*, basta con cambiar ```runs-on``` a ```ubuntu-latest``` y apuntar las variables de conexión a esa máquina. El resto del *workflow* no cambia. Un *runner* hospedado no puede usarse contra el contenedor del *Codespace*.


## Preparación del destino (contenedor *SSH* en el *Codespace*).

En la terminal del *Codespace*, con *Docker* ya disponible:

1. Crear un par de claves solo para esta práctica. El archivo de la clave privada **no** se versiona.

    ```bash
    mkdir -p "$HOME/cd441-destino/ssh"
    ssh-keygen -t ed25519 -f "$HOME/cd441-destino/ssh/deploy" -N ""
    ```

2. Levantar un contenedor que acepte únicamente esa clave pública. Quien imparte el curso puede indicar una imagen concreta; un esquema válido es un servidor *OpenSSH* que lea la clave pública desde una variable de entorno, sin contraseñas en el comando:

    ```bash
    docker run --rm -d \
      --name cd441-destino \
      -e USER_NAME=deploy \
      -e PUBLIC_KEY="$(cat "$HOME/cd441-destino/ssh/deploy.pub")" \
      -p 2222:2222 \
      lscr.io/linuxserver/openssh-server:latest
    ```

    El puerto publicado, el nombre de usuario y la imagen pueden ajustarse. Lo importante es: acceso solo por clave, *Docker* disponible en el destino (o la capacidad de ejecutar el manifiesto de composición) y un directorio de trabajo que se indicará en la variable ```DEPLOY_PATH```.

3. Comprobar, desde el propio *Codespace*:

    ```bash
    ssh -i "$HOME/cd441-destino/ssh/deploy" -p 2222 -o StrictHostKeyChecking=accept-new deploy@127.0.0.1 "uname -a"
    ```

En el repositorio de *GitHub* se configuran:

* el secreto ```SSH_PRIVATE_KEY``` (contenido de la clave privada);
* las variables ```DEPLOY_HOST```, ```DEPLOY_PORT```, ```DEPLOY_USER``` y ```DEPLOY_PATH```.

Con el valor por defecto, ```DEPLOY_HOST``` es ```127.0.0.1``` (el *runner* autoalojado y el contenedor viven en el mismo *Codespace*) y ```DEPLOY_PORT``` es el puerto publicado.


## Controles mínimos.

Un despliegue por *SSH* no es «un comando remoto y ya». En este curso se exige:

* **Claves, no contraseñas**, y la clave privada solo como secreto de *GitHub*, nunca en el repositorio.
* **Verificación del anfitrión** con ```ssh-keyscan``` y el archivo ```known_hosts```, para no aceptar un destino distinto al previsto.
* **Pasos idempotentes**: volver a ejecutar el *workflow* deja el servicio en el mismo estado deseado.
* **Evidencia posterior**: una petición ```GET /health``` en el propio destino.

El *workflow* vigente publica la imagen en *GitHub Container Registry* (*GHCR*) desde la integración continua y, en el destino, actualiza el manifiesto ```api/deploy/compose.yaml``` y levanta el servicio con *Docker Compose*. Los nombres exactos de las variables de ese manifiesto se confirman en el contrato de código del curso.

Una preparación más amplia del anfitrión (por ejemplo, un motor de base de datos en el propio servidor) no es el centro de la explicación de *SSH*; si el manifiesto la incluye, se usa tal como está, sin incrustar contraseñas en el repositorio ni en este cuaderno.


## El *workflow* ```despliega-linux.yml```.

El archivo vigente es ```.github/workflows/despliega-linux.yml```. Se dispara de forma manual (*workflow_dispatch*). Los pasos esenciales, alineados con ese archivo, son:

```yaml
name: Despliegue en Linux por SSH

on:
  workflow_dispatch:

permissions:
  contents: read

jobs:
  deploy:
    # Valor por defecto: runner autoalojado en el Codespace (cuaderno 10).
    # Alternativa con destino alcanzable desde GitHub: cambiar a ubuntu-latest.
    runs-on: self-hosted
    steps:
    - name: Checkout
      uses: actions/checkout@v7

    - name: Configure SSH
      env:
        SSH_PRIVATE_KEY: ${{ secrets.SSH_PRIVATE_KEY }}
        DEPLOY_HOST: ${{ vars.DEPLOY_HOST }}
        DEPLOY_PORT: ${{ vars.DEPLOY_PORT }}
      run: |
        umask 077
        mkdir -p "${HOME}/.ssh"
        printf '%s\n' "${SSH_PRIVATE_KEY}" > "${HOME}/.ssh/id_ed25519"
        ssh-keyscan -p "${DEPLOY_PORT}" "${DEPLOY_HOST}" >> "${HOME}/.ssh/known_hosts"

    - name: Deploy
      env:
        DEPLOY_HOST: ${{ vars.DEPLOY_HOST }}
        DEPLOY_PORT: ${{ vars.DEPLOY_PORT }}
        DEPLOY_USER: ${{ vars.DEPLOY_USER }}
        DEPLOY_PATH: ${{ vars.DEPLOY_PATH }}
        IMAGE: ghcr.io/${{ github.repository }}/api:${{ github.sha }}
        APP_VERSION: ${{ github.sha }}
      run: |
        OPCIONES=(-i "${HOME}/.ssh/id_ed25519" -o IdentitiesOnly=yes -o StrictHostKeyChecking=yes)
        ssh "${OPCIONES[@]}" -p "${DEPLOY_PORT}" "${DEPLOY_USER}@${DEPLOY_HOST}" "mkdir -p '${DEPLOY_PATH}'"
        scp "${OPCIONES[@]}" -P "${DEPLOY_PORT}" api/deploy/compose.yaml "${DEPLOY_USER}@${DEPLOY_HOST}:${DEPLOY_PATH}/compose.yaml"
        ssh "${OPCIONES[@]}" -p "${DEPLOY_PORT}" "${DEPLOY_USER}@${DEPLOY_HOST}" \
          "cd '${DEPLOY_PATH}' && IMAGE='${IMAGE}' APP_VERSION='${APP_VERSION}' docker compose pull && IMAGE='${IMAGE}' APP_VERSION='${APP_VERSION}' docker compose up -d"

    - name: Smoke test
      env:
        DEPLOY_HOST: ${{ vars.DEPLOY_HOST }}
        DEPLOY_PORT: ${{ vars.DEPLOY_PORT }}
        DEPLOY_USER: ${{ vars.DEPLOY_USER }}
      run: |
        OPCIONES=(-i "${HOME}/.ssh/id_ed25519" -o IdentitiesOnly=yes -o StrictHostKeyChecking=yes)
        ssh "${OPCIONES[@]}" -p "${DEPLOY_PORT}" "${DEPLOY_USER}@${DEPLOY_HOST}" \
          "curl --fail --silent --show-error http://127.0.0.1:8080/health"
```

El *workflow* comprueba, al inicio, que el secreto y las variables de conexión estén definidos. La imagen se identifica con el *SHA* del *commit* en *GHCR*. La ruta de salud es ```GET /health```.


## Actividad.

1. En el *Codespace*, preparar el destino por defecto (contenedor *SSH* y *runner* autoalojado del cuaderno 10) o, si se usa la alternativa, el destino indicado por quien imparte el curso.
2. Cargar el secreto ```SSH_PRIVATE_KEY``` y las variables ```DEPLOY_HOST```, ```DEPLOY_PORT```, ```DEPLOY_USER``` y ```DEPLOY_PATH``` en el repositorio propio.
3. Disparar el *workflow* ```Despliegue en Linux por SSH``` y comprobar que ```GET /health``` responde en el destino.
4. Volver a dispararlo sin cambiar el código y verificar que el resultado es el mismo (idempotencia).
5. Localizar en el *workflow* la línea ```runs-on``` que habría que cambiar para pasar del *runner* autoalojado del *Codespace* a un *runner* hospedado (solo válida con la alternativa).


## Cierre.

Con este cuaderno se cierra el tramo práctico de *cd441*, trabajado por completo en *GitHub Codespaces*: integración continua de una *API*, imagen en un registro, un primer despliegue en *Cloud Run* y un primer despliegue en *Linux* por *SSH*. Los cursos posteriores pueden dar esa práctica por cubierta y avanzar hacia infraestructura como código, federación de identidad y servicios de *GCP* que aquí solo se nombraron.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023.</p>